# Data injection

In [1]:
import os
import yt_dlp #to download videos from youtube
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from dotenv import load_dotenv # to load environment variables from a .env file
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

C:\Users\sharddha\AppData\Local\Temp\ipykernel_19940\3034944156.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name = 'all-MiniLM-L6-v2')

URL Extraction Function using yt-dlp

In [3]:
# ==========================================
#  DATA INGESTION 
# ==========================================

def load_youtube_videos(playlist_url : str) -> list: 
    video_urls= []
    ydl_opts = {
        'extract_flat' : True,  #extract video urls without downloading the videos
        'quiet' : True,  #suppress output
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            playlist_info = ydl.extract_info(playlist_url, download=False)
            if 'entries' in playlist_info:
                for entry in playlist_info['entries']:
                    if entry:
                        vid_id = entry.get('id')  ## yt-dlp 'extract_flat' usually stores the video ID under 'id'
                        if vid_id:
                            video_url = f"https://www.youtube.com/watch?v={vid_id}"
                            video_urls.append(video_url)
                        
        except Exception as e:
            print(f"Error loading playlist: {playlist_url}. Error: {e}")

    return video_urls



# Document Loading Function using youtubetranscriptapi

In [ ]:
# ================================
# Data Processing
# ================================

import requests
import http.cookiejar

def load_videos(video_urls_list):
    loaded_docs = []
    # Check if we actually found videos
    if not video_urls_list:
        print("No video URLs found to process.")
        return loaded_docs
    
    
    session = requests.Session()
    try:
        cookie_jar = http.cookiejar.MozillaCookieJar('cookies.txt')
        cookie_jar.load(ignore_discard=True, ignore_expires=True)
        session.cookies.update(cookie_jar)
    except Exception as e:
        print(f"Warning: Could not load cookies: {e}")

    api = YouTubeTranscriptApi(http_client=session)
   
    for video_url in video_urls_list:
        try:      # Extract video ID from URL
            if "v=" in video_url:
                video_id = video_url.split("v=")[1].split("&")[0] # Extract video ID from URL
            elif "youtu.be/" in video_url:                        
                video_id = video_url.split("youtu.be/")[1].split("?")[0]
            else:
                print(f"Invalid video URL format: {video_url}")
                continue
              
            fetched_transcript = api.fetch(
                video_id,
                languages=["en", "hi", "en-IN"]
            )
            transcript = [{"text": snippet.text} for snippet in fetched_transcript]
                        
            full_transcript_text = " ".join([item['text'] for item in transcript])
            print(fetched_transcript)
            doc =  Document(page_content = full_transcript_text,
                            metadata = {'source': video_url,
                                        'pages': 1,
                                        'title': f'Transcript for {video_id}',
                                        'author': 'YouTube'
                                        })
            loaded_docs.append(doc)
            print(f"Successfully loaded text for: {video_url}")
                
        except Exception as e:
            print(f"Skipping video {video_url} due to error (e.g. no English/hindi transcripts): {e}")
    
    print('='*50)
    return loaded_docs

In [5]:
#playlist_url = 'https://www.youtube.com/playlist?list=PLZoTAELRMXVM8Pf4U67L4UuDRgV4TNX9D'
def process_youtube_videos(my_url : str):
    # checking if url is a playlist or a single video
    if 'list=' in my_url:
        video_urls = load_youtube_videos(my_url)
    else:
        print('Single video detected!')
        video_urls = [my_url]

    all_loaded_docs = load_videos(video_urls)
    all_loaded_docs
    print(f"\nTotal Documents Loaded: {len(all_loaded_docs)}")

    return all_loaded_docs

#all_loaded_docs_withtrans = process_youtube_videos(playlist_url)

# Chunking & Embedding

In [ ]:
# ==========================================
# 2. CHUNKING & EMBEDDING
# ==========================================

def generate_embedding(all_loaded_docs, persist_directiory = './faiss_db'):
    if not all_loaded_docs:
            print("No documents to embed!")
            return None
    #chunking
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    text_chunks = text_splitter.split_documents(all_loaded_docs)
    print(f"Split {len(all_loaded_docs)} original documents into {len(text_chunks)} smaller chunks.")
    print(text_chunks[0])
    
    print('*'*50)

    #text-embedding-3-small is OpenAI's newest, cheapest, and most efficient model)
    embedding_model = HuggingFaceEmbeddings(model_name = 'all-MiniLM-L6-v2') 

    vectorstore = FAISS.from_documents(
            documents= text_chunks,
            embedding= embedding_model
            )
    # Save the FAISS database locally
    vectorstore.save_local(persist_directiory)
    print(f"Success! Your text is now embedded and stored locally in a Vector Database named {persist_directiory}.")
    vectorstore.index_to_docstore_id
    # vectorstore.get_by_ids([''])
    return vectorstore


#my_vector_database = generate_embedding(all_loaded_docs_withtrans,"./faiss_db")

#my_vector_database

# Retrievel

In [ ]:
def retriever_func(query: str) ->list:
    embedding_model = HuggingFaceEmbeddings(model_name = 'all-MiniLM-L6-v2')
    # load the saved vector store
    persist_directory = './faiss_db'
    vector_store = FAISS.load_local(
        folder_path= persist_directory,
        embeddings=embedding_model,
        allow_dangerous_deserialization= True   # Required by LangChain to load local pickle files
    )
    #finding the top 4 relevent vectors from database related to query
    retriever = vector_store.as_retriever(search_type = 'similarity', search_kwargs = {'k':4})
    docs = retriever.invoke(query)
    return docs


# query = 'what is LLM?'
# reterived_docs = retriever_func(query)   

In [8]:
# ==========================================
# 3. RAG GENERATION WITH LCEL (RUNNABLES)
# ==========================================

def format_doc(docs):
    return "\n\n".join(
        f"Source: {doc.metadata.get('source', 'Unknown Website')}\n"
        f"Content: {doc.page_content}"
        for doc in docs
    )

In [ ]:
def generate_answers(query):
    # 1. Create the Prompt Template
    prompt = PromptTemplate(
        template="""
    You are an expert AI assistant. Your task is to provide a clear, accurate, and concise answer to the user's question.

    Instructions:
    - Primary Answer: Base your main answer on the provided Context. Always share the content source (URL or title) provided in the context.
    - Handling Incomplete Data: If the context does not contain enough information to fully answer the question, you may provide additional relevant details from your general knowledge.
    - Outsourcing Disclaimer: If you include information that is NOT found in the context, you MUST explicitly state: "Note: This part is outsourcing from external knowledge/websites to give you the full content knowledge."
    - Format: Use bullet points for readability and keep the answer concise.

    Context:
    {context}

    Question: {query}
    """,
        input_variables=['context', 'query']
    )

    # 4. Initialize Gemini
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash", 
        temperature=0.3, 
        max_tokens=1024
    )
    # Parallel retrieval chain
    setup_and_retrieval = RunnableParallel(
            context = RunnableLambda(retriever_func) | RunnableLambda(format_doc),
            query = RunnablePassthrough()
            )
    #full execution
    rag_chain = setup_and_retrieval | prompt | llm | StrOutputParser()

    # 5. Invoke the LLM
    print(f"Asking the LLM: {query}\n")
    response = rag_chain.invoke(query)

    # 6. Print and return the clean output
    print("$$$ FINAL ANSWER $$$")
    
    
    return response


In [13]:
# ==========================================
# FINAL EXECUTION
# ==========================================
if __name__ == "__main__":
    # Load environment variables
    load_dotenv("key.env")
    gemini_key = os.getenv("GEMINI_API_KEY")

    if not gemini_key:
        print("WARNING: Please ensure GEMINI_API_KEY is set in your key.env file.")
    else:
        os.environ["GOOGLE_API_KEY"] = gemini_key

   

    # Ask the user for the YouTube URL
    user_url = input("Enter the YouTube Video or Playlist URL: ").strip()
    print("\nLoading transcripts... This might take a moment.")

    # Process and download transcripts
    all_loaded_docs_withtrans = process_youtube_videos(user_url)

    # Generate Vector DB
    if all_loaded_docs_withtrans:
        print("\nCreating the Vector Database...")
        my_vector_database = generate_embedding(all_loaded_docs_withtrans, "./faiss_db")

 
    print("\n" + "="*50)
    print("Database Ready! Ask questions about the video(s).")
    print("Type 'exit' or 'quit' to stop.")
    print("="*50 + "\n")

    # Start the Interactive Q&A While Loop
    while True:
            # Ask for the user's question
        query = input("Your Question: ").strip()
        
        #pass the query and the chunks directly into generator function
        if query.lower() in ['exit', 'quit', 'q']:
            print("Ending the session. Goodbye!")
            break

        if not query:
            continue

        final_answer = generate_answers(query)
        print(final_answer)
        print('*'*50)



Loading transcripts... This might take a moment.
FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='Hello all, my name is Krishna and', start=0.48, duration=4.24), FetchedTranscriptSnippet(text='welcome to my YouTube channel. So guys,', start=2.56, duration=4.48), FetchedTranscriptSnippet(text='I am super excited to start this new', start=4.72, duration=5.2), FetchedTranscriptSnippet(text='series on one of the most important', start=7.04, duration=4.88), FetchedTranscriptSnippet(text='technique which is right now being used', start=9.92, duration=5.04), FetchedTranscriptSnippet(text='in genative AI and agentic AI field that', start=11.92, duration=5.279), FetchedTranscriptSnippet(text="is nothing but rag. If you don't know", start=14.96, duration=3.76), FetchedTranscriptSnippet(text='the full form of rag, it is called as', start=17.199, duration=4.24), FetchedTranscriptSnippet(text='retrieval augmented generation. In this', start=18.72, duration=4.479), FetchedTranscriptSnippe

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3578.08it/s]


Success! Your text is now embedded and stored locally in a Vector Database named ./faiss_db.

Database Ready! Ask questions about the video(s).
Type 'exit' or 'quit' to stop.



Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Asking the LLM: why rag?



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1794.01it/s]


$$$ FINAL ANSWER $$$
Here are the reasons for using RAG:

*   **Overcomes LLM Disadvantages:** RAG helps to overcome the disadvantages of just using Large Language Models (LLMs) alone.
*   **Optimizes LLM Output:** It optimizes the output of an LLM by referencing an authoritative knowledge base outside of its training data before generating a response.
*   **Extends LLM Capabilities:** RAG extends the powerful capabilities of LLMs to specific domains or an organization's internal knowledge base without the need to retrain the model.
*   **Cost-Effective:** It is a cost-effective approach to improve LLM output, making it more relevant, accurate, and useful in various contexts.
*   **Prevents Frequent Fine-tuning:** RAG provides a solution to prevent the need for frequent fine-tuning of LLMs.
*   **High Industry Demand:** It is a highly sought-after technique in generative AI and agentic AI, with many companies (around 60-70% of projects) focusing on RAG applications.

Source: https://ww